# Fase 3: Validación de la Capacidad Operativa
**Objetivo:** Verificar que la flota actual de 60 camiones CAT 797F es capaz de cumplir con las metas de producción bajo restricciones operativas reales, específicamente ajustando la disponibilidad mecánica al 83% según las instrucciones del caso (saneamiento de datos).

### 1. Saneamiento de Parámetros y Flota Activa
El archivo base de parámetros indicaba un 85% de disponibilidad, sin embargo, las reglas de negocio estrictas exigen modelar al 83%. Ajustaremos esto para obtener la flota real que estará moviendo mineral.

In [14]:
import pandas as pd
import numpy as np

# PARÁMETROS OPERATIVOS SANEADOS
# Corrección según instrucciones PDF: Disponibilidad mecánica = 83%
disp_mecanica_corregida = 0.83
flota_total = 60
horas_efectivas_dia = 21.83  # Dato extraído del CSV de parámetros
carga_operativa_ton = 320    # Capacidad real por viaje en toneladas

# Cálculo de la flota real disponible para operar
flota_activa = flota_total * disp_mecanica_corregida

print("SANEAMIENTO DE FLOTA")
print(f"Flota física total: {flota_total} camiones")
print(f"Flota activa real (al 83% de disponibilidad): {flota_activa:.2f} camiones")

SANEAMIENTO DE FLOTA
Flota física total: 60 camiones
Flota activa real (al 83% de disponibilidad): 49.80 camiones


### 2. Modelamiento Cinemático y Tiempos de Ciclo
Para saber cuánto puede mover la flota, necesitamos calcular cuánto demora un camión en ir y volver de cada tajo. 
* **Tajo Norte:** El camión sube cargado (+10% pendiente), lo que lo hace más lento de ida, pero baja rápido vacío.
* **Tajo Sur:** El camión baja cargado (-10% pendiente). Esto hace el ciclo más rápido por gravedad, pero exige un uso extremo de los frenos/retardadores, lo que explica térmicamente el mayor desgaste de neumáticos descubierto en la Fase 1.

In [15]:
# TIEMPOS DE CICLO (Estimación técnica CAT 797F)
# Tajo Norte: 4.5 km
# Tajo Sur: 3.8 km

# Velocidades estimadas (km/h) asumidas por catálogo de equipo pesado
vel_subida_cargado = 13.5
vel_bajada_vacio = 35.0
vel_bajada_cargado = 22.0
vel_subida_vacio = 28.0

# Tiempo muerto: Cuadre en pala, carga, descarga y maniobras (asumimos 4.5 minutos)
tiempo_cuadra_carga_descarga = 4.5 / 60  # Convertido a horas

# Ciclo Tajo Norte (Sube cargado, baja vacío)
tiempo_norte = (4.5 / vel_subida_cargado) + (4.5 / vel_bajada_vacio) + tiempo_cuadra_carga_descarga
viajes_hora_norte = 1 / tiempo_norte
prod_hora_norte = viajes_hora_norte * carga_operativa_ton

# Ciclo Tajo Sur (Baja cargado, sube vacío)
tiempo_sur = (3.8 / vel_bajada_cargado) + (3.8 / vel_subida_vacio) + tiempo_cuadra_carga_descarga
viajes_hora_sur = 1 / tiempo_sur
prod_hora_sur = viajes_hora_sur * carga_operativa_ton

print("RENDIMIENTO POR TAJO")
print(f"Producción Tajo Norte: {prod_hora_norte:.2f} ton/hora por camión")
print(f"Producción Tajo Sur: {prod_hora_sur:.2f} ton/hora por camión")

RENDIMIENTO POR TAJO
Producción Tajo Norte: 596.01 ton/hora por camión
Producción Tajo Sur: 834.55 ton/hora por camión


### 2.1 Visualización del Rendimiento Operacional por Tajo

A continuación, se presenta una comparación gráfica de la capacidad de producción horaria por camión entre el Tajo Norte y el Tajo Sur.

Esta visualización permite identificar el impacto operacional de:

- la geometría de acarreo,
- las pendientes,
- las velocidades operativas,
- y los tiempos de ciclo

sobre la productividad de la flota CAT 797F.

In [16]:
import pandas as pd
import plotly.express as px

# DATAFRAME

df_prod = pd.DataFrame({ "Tajo": ["Tajo Norte","Tajo Sur"],"Produccion_ton_h": [prod_hora_norte,prod_hora_sur]})


# DIFERENCIA %

incremento_prod = ((prod_hora_sur -prod_hora_norte)/prod_hora_norte) * 100


# GRÁFICO

fig = px.bar(

    df_prod,
    x="Tajo",
    y="Produccion_ton_h",
    text="Produccion_ton_h",
    title="Capacidad Operacional de Transporte por Tajo",
    template="plotly_white"
)


# PERSONALIZACIÓN

fig.update_traces(

    texttemplate="%{text:.2f} ton/h",
    textposition="outside"
)

fig.update_layout(

    title_x=0.5,
    title_font_size=24,
    xaxis_title="Zona Operacional",
    yaxis_title="Producción (ton/hora por camión)",
    font=dict(size=15),height=650,width=1000
)


# ANOTACIÓN EJECUTIVA

fig.add_annotation(

    x="Tajo Sur",
    y=prod_hora_sur,
    text=(f"<b>{incremento_prod:.1f}%</b><br>"f"más productividad"),
    showarrow=True,
    arrowhead=2,
    yshift=35,
    bordercolor="black",
    borderwidth=1,
    bgcolor="white"
)

# MOSTRAR

fig.show()

### 3. Capacidad Máxima del Sistema de Transporte
Sabiendo cuánto produce un camión en cada ruta y cuántos camiones tenemos disponibles al día (flota activa), proyectaremos la capacidad de movimiento diario y mensual para validar si el sistema es robusto. Para este escenario base de validación, asumiremos una distribución 50/50 de la flota.

In [17]:
# CÁLCULO DE CAPACIDAD TOTAL

# Asumimos un split equilibrado de la flota activa para la validación base
camiones_por_tajo = flota_activa / 2

# Producción total = (Camiones Norte * Prod Norte * Horas al día) + (Camiones Sur * Prod Sur * Horas al día)
produccion_diaria_total = (camiones_por_tajo * prod_hora_norte * horas_efectivas_dia) +(camiones_por_tajo * prod_hora_sur * horas_efectivas_dia)

produccion_mensual_total = produccion_diaria_total * 28 # Según parámetros: 28 días de operación/mes

print("=== RESULTADO DE LA VALIDACIÓN OPERATIVA ===")
print(f"Capacidad de movimiento diario: {produccion_diaria_total:,.2f} Toneladas/día")
print(f"Capacidad de movimiento mensual: {produccion_mensual_total:,.2f} Toneladas/mes")

=== RESULTADO DE LA VALIDACIÓN OPERATIVA ===
Capacidad de movimiento diario: 777,602.96 Toneladas/día
Capacidad de movimiento mensual: 21,772,882.85 Toneladas/mes


### 3.1 Indicadores Ejecutivos de Capacidad Operacional

Para representar la capacidad operacional de forma más ejecutiva y alineada a dashboards industriales tipo Power BI, se utilizarán indicadores KPI en formato cápsula.

Estos indicadores permiten visualizar rápidamente:

- capacidad diaria,
- capacidad mensual,
- y validación operacional de la flota.

In [18]:
import plotly.graph_objects as go

# FIGURA KPI

fig = go.Figure()

# KPI PRODUCCIÓN DIARIA

fig.add_trace(

    go.Indicator(

        mode="number",
        value=produccion_diaria_total,
        title={"text":"<b>Producción Diaria</b><br>""<span style='font-size:16px'>Toneladas/día</span>"},
        number={"valueformat": ",.0f"},
        domain={"x": [0.0, 0.45],"y": [0, 1]}
    )
)

# KPI PRODUCCIÓN MENSUAL

fig.add_trace(

    go.Indicator(

        mode="number",

        value=produccion_mensual_total,

        title={
            "text":
            "<b>Producción Mensual</b><br>"
            "<span style='font-size:16px'>Toneladas/mes</span>"
        },

        number={"valueformat": ",.0f"},

        domain={"x": [0.55, 1],"y": [0, 1]}
    )
)

# LAYOUT
fig.update_layout(
title={

        "text":
        "KPIs de Validación Operativa de la Flota CAT 797F",
        "x": 0.5,
        "font": {"size": 24}
    },
    template="simple_white",
    height=400,
    width=1000,
    font=dict(
        size=18
    ),
    margin=dict(
        t=100,
        b=40,
        l=40,
        r=40
    )
)
# MOSTRAR
fig.show()

### 4. Conclusión Operativa
* **Validación Positiva:** La capacidad demostrada (más de 21 millones de toneladas mensuales) confirma que una flota de 60 equipos operando al 83% de disponibilidad es más que suficiente para sostener una operación de gran minería.
* **Correlación Geomecánica:** Queda evidenciado que el Tajo Sur tiene ciclos más rápidos (834 ton/h vs 596 ton/h del Norte) debido a que los camiones bajan cargados. Esta mayor velocidad y peso en descenso aumenta drásticamente el TKPH (Toneladas-Kilómetro Por Hora) térmico de la llanta, validando los resultados estadísticos de la Fase 1: **El Tajo Sur destruye las llantas más rápido por fatiga térmica y frenado.**